In [ ]:
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 5 Lab — Supervised Learning: Classification
**Introduction to Data Science — DBAS 5015**

---

<!-- ============================================================
  WRITING STYLE — apply to all markdown cells in this notebook
  - Direct and to the point. Say what the concept is and move on.
  - No hedging. State things plainly.
  - Active voice. 'The function returns a value' not 'A value is returned.'
  - Short sentences for instructions. Students need to scan them quickly.
  - Exercise instructions must be unambiguous — one reading, one meaning.
  - No filler. Every sentence should add information or cut it.
  ============================================================ -->

### What This Lab Covers
- Exploring a binary classification dataset and its class balance
- Fitting `LogisticRegression` and evaluating with a confusion matrix and classification report
- Fitting `DecisionTreeClassifier` and reading feature importances
- Comparing two models on the same metric

**Estimated time:** 80–95 minutes

---

## The Dataset

This lab uses a synthetic customer churn dataset. Each row is a customer account. The target is `churned` — 1 if the customer cancelled in the period, 0 if they stayed. The goal is to predict which customers are at risk of churning before they leave.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, f1_score, precision_score, recall_score
)

np.random.seed(13)
n = 400

tenure       = np.random.randint(1, 73, n)
monthly_chg  = np.round(np.random.uniform(20, 120, n), 2)
num_products = np.random.randint(1, 6, n)
tech_support = np.random.choice([0, 1], n, p=[0.45, 0.55])
contract     = np.random.choice(
    ['Month-to-Month', 'One Year', 'Two Year'], n, p=[0.50, 0.30, 0.20]
)
segment      = np.random.choice(
    ['Residential', 'SMB', 'Enterprise'], n, p=[0.55, 0.30, 0.15]
)

# Churn probability — shorter tenure, higher charges, month-to-month → higher risk
contract_risk = np.where(contract == 'Month-to-Month', 0.35,
               np.where(contract == 'One Year', 0.10, 0.03))
segment_risk  = np.where(segment == 'Residential', 0.08,
               np.where(segment == 'SMB', 0.03, 0.00))

churn_prob = (
    0.45 - (tenure / 72) * 0.35
    + (monthly_chg - 20) / 100 * 0.20
    - tech_support * 0.10
    - (num_products - 1) / 4 * 0.08
    + contract_risk
    + segment_risk
)
churn_prob = np.clip(churn_prob, 0.02, 0.95)
churned = (np.random.uniform(0, 1, n) < churn_prob).astype(int)

df = pd.DataFrame({
    'tenure_months':  tenure,
    'monthly_charges': monthly_chg,
    'num_products':   num_products,
    'tech_support':   tech_support,
    'contract_type':  contract,
    'customer_segment': segment,
    'churned':        churned,
})

print(f'Dataset ready: {df.shape[0]} rows, {df.shape[1]} columns')
print(df.dtypes)

---
## Part 1 — Explore the Dataset and Class Balance

Classification requires understanding the target distribution before fitting anything. A model trained on imbalanced classes without awareness of that imbalance will optimize for the wrong thing.

In [ ]:
# Class balance
churn_counts = df['churned'].value_counts()
churn_rate   = df['churned'].mean()

print('Churned counts:')
print(churn_counts)
print(f'\nChurn rate: {churn_rate:.1%}')

# Churn rate by contract type
print('\nChurn rate by contract type:')
print(df.groupby('contract_type')['churned'].mean().round(3))

# Churn rate by customer segment
print('\nChurn rate by customer segment:')
print(df.groupby('customer_segment')['churned'].mean().round(3))

---
### 💼 Business Context — The Baseline Model

Before building any model, establish the **baseline accuracy**: the accuracy of the dumbest possible model, which always predicts the majority class. If 75% of customers are retained, predicting 'retained' for everyone achieves 75% accuracy. Any model you build must beat this baseline — otherwise it has learned nothing. Report baseline accuracy alongside model accuracy so stakeholders understand what 'improvement' means.

---

### ✏️ Exercise 1.1 — EDA and Baseline

1. Compute the baseline accuracy — the accuracy of always predicting the majority class (0 — retained).
2. Plot a histogram of `tenure_months` for churned customers (churned = 1) vs. retained customers (churned = 0) on the same axes using `plt.hist` with `alpha=0.6`. This shows whether tenure differs between the two groups.
3. Print the mean `monthly_charges` for churned vs. retained customers.
4. Separate the dataset into `X` (all columns except `churned`) and `y` (`churned`).

In [ ]:
# Exercise 1.1

# 1. Baseline accuracy
baseline_acc = 
print(f'Baseline accuracy (always predict retained): {baseline_acc:.1%}')

# 2. Tenure distribution — churned vs. retained
churned_tenure  = df[df['churned'] == 1]['tenure_months']
retained_tenure = df[df['churned'] == 0]['tenure_months']

plt.figure(figsize=(8, 3))
# your code here
plt.xlabel('Tenure (months)')
plt.ylabel('Count')
plt.title('Tenure Distribution — Churned vs. Retained')
plt.legend(['Churned', 'Retained'])
plt.tight_layout()
plt.show()

# 3. Mean monthly charges by churn status
mean_charges = 
print('\nMean monthly charges:')
print(mean_charges)

# 4. Separate features and target
X = 
y = 
print(f'\nX shape: {X.shape}, y shape: {y.shape}')

---
## Part 2 — Preprocess and Split

This dataset has three types of columns requiring different treatment:

| Column | Type | Preprocessing |
|:-------|:-----|:--------------|
| `contract_type` | Ordinal (Month-to-Month < One Year < Two Year) | `OrdinalEncoder` |
| `customer_segment` | Nominal | `OneHotEncoder` |
| `tenure_months`, `monthly_charges`, `num_products` | Continuous numeric | `StandardScaler` |
| `tech_support` | Binary flag | `passthrough` |

In [ ]:
# Column setup
ordinal_cols    = ['contract_type']
categorical_cols = ['customer_segment']
numerical_cols  = ['tenure_months', 'monthly_charges', 'num_products']
passthrough_cols = ['tech_support']

# Build the preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('ord',  OrdinalEncoder(categories=[['Month-to-Month', 'One Year', 'Two Year']]),
             ordinal_cols),
    ('cat',  OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False),
             categorical_cols),
    ('num',  StandardScaler(), numerical_cols),
    ('pass', 'passthrough', passthrough_cols),
])

# Split — stratify on y to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit on training data only, transform both
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f'Training set: {X_train_processed.shape}')
print(f'Test set:     {X_test_processed.shape}')
print(f'\nChurn rate — y_train: {y_train.mean():.1%}  y_test: {y_test.mean():.1%}')

---
### 🤖 ML Connection — Why `stratify=y` Matters More for Imbalanced Data

With balanced classes (50/50), a random split will almost always produce representative subsets. With imbalanced classes (e.g., 25% churn), a random split can produce a test set with 18% or 32% churn by chance — both would give misleading evaluation results. `stratify=y` guarantees the split preserves the original class proportion in both sets. Use it for every classification problem.

---

### ✏️ Exercise 2.1 — Verify Preprocessing

1. Print all output column names from `preprocessor.get_feature_names_out()`.
2. Count how many columns came from each transformer (ordinal, categorical, numeric, passthrough).
3. Confirm the `contract_type` encoding is correct: print the first 5 values of the ordinal column alongside the original `contract_type` values from `X_train`.

In [ ]:
# Exercise 2.1

# 1. All output column names
output_cols = preprocessor.get_feature_names_out()
print('Output columns:')
print(output_cols)

# 2. Count by transformer
ord_count  = 
cat_count  = 
num_count  = 
pass_count = 
print(f'\nOrdinal: {ord_count}  |  Categorical OHE: {cat_count}  |  Numeric: {num_count}  |  Passthrough: {pass_count}')

# 3. Verify contract_type ordinal encoding
check = pd.DataFrame({
    'original': X_train['contract_type'].values[:5],
    'encoded':  X_train_processed[:5, 0]   # ordinal column is first
})
print('\nContract type encoding check:')
print(check)

---
## Part 3 — Logistic Regression

Logistic regression is the baseline classification model. Fit it first — if a more complex model does not beat logistic regression, the added complexity is not justified.

In [ ]:
# Fit logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_processed, y_train)

# Predict
y_pred_lr = lr.predict(X_test_processed)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retained', 'Churned'])
disp.plot(cmap='Blues')
plt.title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

# Classification report
print(classification_report(y_test, y_pred_lr, target_names=['Retained', 'Churned']))

---
### 🤖 ML Connection — Reading the Classification Report

Read the report row by row for each class, then look at the weighted average. For a churn model, the Churned row is the one that matters. Precision tells you how many of the customers you flagged actually churned — the false alarm rate for your retention team. Recall tells you how many actual churners you caught — your miss rate. The weighted average blends both classes by their support (count), which gives it a class-imbalance-aware summary. Accuracy at the bottom of the report is presented for context only — do not use it as your primary metric.

---

### ✏️ Exercise 3.1 — Evaluate and Interpret

1. From the classification report, record the precision, recall, and F1 for the `Churned` class.
2. Compute the four confusion matrix values (TP, TN, FP, FN) from the `cm` array. Print each with a label.
3. The model has a churn recall below 1.0. Write a comment: what does each missed churner cost the business, and what recall rate would you recommend targeting for a retention campaign?
4. Check: does this model beat the baseline accuracy from Exercise 1.1?

In [ ]:
# Exercise 3.1

# 1. Precision, recall, F1 for Churned class
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall    = 
lr_f1        = 
print(f'Logistic Regression — Churned class:')
print(f'  Precision: {lr_precision:.3f}')
print(f'  Recall:    {lr_recall:.3f}')
print(f'  F1:        {lr_f1:.3f}')

# 2. TP, TN, FP, FN from confusion matrix
# cm[0,0]=TN, cm[0,1]=FP, cm[1,0]=FN, cm[1,1]=TP
TN = 
FP = 
FN = 
TP = 
print(f'\nConfusion matrix breakdown:')
print(f'  True Positives  (TP): {TP}  — churners correctly flagged')
print(f'  True Negatives  (TN): {TN}  — retained customers correctly identified')
print(f'  False Positives (FP): {FP}  — false alarms (predicted churn, stayed)')
print(f'  False Negatives (FN): {FN}  — missed churners')

# 3. Business cost of missed churners
# 
# 

# 4. Does the model beat baseline?
model_acc = (TP + TN) / (TP + TN + FP + FN)
print(f'\nModel accuracy:    {model_acc:.1%}')
print(f'Baseline accuracy: {baseline_acc:.1%}')

---
## Part 4 — Decision Tree and Model Comparison

Fit a decision tree on the same preprocessed data and compare its performance to logistic regression. Decision trees offer native feature importance — a different view of which features drive predictions.

In [ ]:
# Fit decision tree
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train_processed, y_train)

# Predict
y_pred_tree = tree.predict(X_test_processed)

# Classification report
print('Decision Tree:')
print(classification_report(y_test, y_pred_tree, target_names=['Retained', 'Churned']))

# Feature importances
importances = pd.DataFrame({
    'feature':    preprocessor.get_feature_names_out(),
    'importance': tree.feature_importances_
}).sort_values('importance', ascending=False)

print('Feature importances:')
print(importances.to_string(index=False))

---
### 💼 Business Context — Feature Importance vs. Coefficients

Logistic regression coefficients show the direction and magnitude of each feature's effect on the log-odds of churn — positive means the feature increases churn risk. Decision tree feature importances show how much of the total information gain came from each feature — but they do not tell you the direction. A feature with high importance could increase or decrease churn risk depending on its value. Use both tools together: tree importance to identify which features matter, coefficients to understand how they matter.

---

### ✏️ Exercise 4.1 — Compare Models and Visualize Importances

1. Compute F1 for the Churned class for both models. Print the comparison.
2. Which model has higher recall for the Churned class? Recall matters most here — the business goal is catching as many churners as possible.
3. Create a horizontal bar chart of the decision tree's feature importances, sorted descending.
4. Identify the top 2 features by importance and write a one-sentence business interpretation of each.

In [ ]:
# Exercise 4.1

# 1. F1 comparison
tree_f1 = f1_score(y_test, y_pred_tree)
print(f'Logistic Regression F1 (Churned): {lr_f1:.3f}')
print(f'Decision Tree F1 (Churned):       {tree_f1:.3f}')

# 2. Recall comparison
tree_recall = recall_score(y_test, y_pred_tree)
print(f'\nLogistic Regression Recall: {lr_recall:.3f}')
print(f'Decision Tree Recall:       {tree_recall:.3f}')
# Which is better for this business problem?
# 

# 3. Feature importance chart
sorted_imp = importances.sort_values('importance')

plt.figure(figsize=(8, 4))
plt.barh(sorted_imp['feature'], sorted_imp['importance'], color='#4575b4')
plt.xlabel('Importance')
plt.title('Decision Tree Feature Importances — Churn Model')
plt.tight_layout()
plt.show()

# 4. Interpret the top 2 features
top2 = importances.head(2)
print('\nTop 2 features:')
print(top2.to_string(index=False))
# Feature 1 interpretation:
# 
# Feature 2 interpretation:
# 

---
## Challenge Exercise

This section is optional — attempt it if you finish early or want a stretch.

### Adjusting the Decision Threshold

By default, `predict()` classifies a customer as churned if the predicted probability exceeds 0.5. Lowering this threshold flags more customers as at-risk — increasing recall at the cost of precision.

1. Use `lr.predict_proba(X_test_processed)[:, 1]` to get the predicted churn probability for each test customer.
2. Apply a threshold of **0.35** instead of 0.5 to produce a new set of predictions.
3. Print a classification report using the 0.35-threshold predictions.
4. Compare precision and recall for the Churned class at both thresholds (0.5 and 0.35).
5. **Reflection:** If the retention team can contact 40 customers per week and a false alarm (contacting a non-churner) costs $5 in effort, while a missed churner costs $120 in lost revenue — which threshold produces better business outcomes?

In [ ]:
# Challenge — decision threshold adjustment

# 1. Predicted probabilities
y_proba = lr.predict_proba(X_test_processed)[:, 1]
print(f'Probability range: {y_proba.min():.3f} — {y_proba.max():.3f}')

# 2. Apply threshold of 0.35
threshold = 0.35
y_pred_035 = 

# 3. Classification report at 0.35 threshold
print('Classification report at threshold = 0.35:')
print(classification_report(y_test, y_pred_035, target_names=['Retained', 'Churned']))

# 4. Precision and recall comparison
print(f'Threshold 0.50 — Precision: {lr_precision:.3f}  Recall: {lr_recall:.3f}')
print(f'Threshold 0.35 — Precision: {precision_score(y_test, y_pred_035):.3f}  '
      f'Recall: {recall_score(y_test, y_pred_035):.3f}')

# 5. Business calculation
# At threshold 0.50:
#   FP = (from cm), FN = (from cm)
#   Cost = FP * $5 + FN * $120
# At threshold 0.35:
#   FP = ?, FN = ?
#   Cost = FP * $5 + FN * $120
cm_035 = confusion_matrix(y_test, y_pred_035)
FP_035 = cm_035[0, 1]
FN_035 = cm_035[1, 0]

cost_050 = FP * 5 + FN * 120
cost_035 = FP_035 * 5 + FN_035 * 120

print(f'\nBusiness cost at threshold 0.50: ${cost_050:,}')
print(f'Business cost at threshold 0.35: ${cost_035:,}')
# Reflection:
# 

---
## Week 5 Summary

| Concept | Key Takeaway |
|:--------|:-------------|
| Baseline accuracy | Always predict majority class — any model must beat this |
| `stratify=y` | Preserves class proportions in both train and test — critical for imbalanced targets |
| `LogisticRegression(max_iter=1000)` | Baseline classifier — fast, interpretable, well-calibrated probabilities |
| `DecisionTreeClassifier(max_depth=4)` | Nonlinear, interpretable splits — set `max_depth` to prevent overfitting |
| Confusion matrix | TP/TN/FP/FN — the full breakdown of prediction outcomes |
| Precision | Of predicted positives, how many are correct — optimise when false alarms are costly |
| Recall | Of actual positives, how many were caught — optimise when misses are costly |
| F1 score | Harmonic mean — use when both precision and recall matter |
| `classification_report` | One call, all metrics — read the row for your positive class |
| Feature importance | Tree-native measure — high importance = feature used heavily in splits |
| Decision threshold | Lower threshold → more positives predicted → recall ↑, precision ↓ |
| Model comparison | Use the same metric on the same test set — compare F1 or recall for the positive class |

---
**Next week:** Model tuning — random forests, cross-validation, and hyperparameter search. You will learn why a single train/test split can be misleading and how to find the right `max_depth` systematically.